# Building Complex Dimers from Multiple PDBs

**Task:** Create a clean, repaired, and aligned protein-protein complex by harvesting monomers from different sources.

In this recipe, we take the structure with PDB id [1BRS](https://www.rcsb.org/structure/1BRS). It contains three barnase-barstar complexes, but they are incomplete. We will "fish" the best monomers, align them, and build a perfect dimer from scratch.

In [ ]:
import molsysmt as msm
import warnings
warnings.filterwarnings('ignore')

### 1. Harvesting the Monomers
We download the system and identify the best chains (B and F are the most complete).

In [ ]:
system = msm.convert('pdb_id:1BRS')

# Extract specific monomers
barnase = msm.extract(system, selection="chain_name=='B'")
barstar_E = msm.extract(system, selection="chain_name=='E'") # Reference position
barstar_F = msm.extract(system, selection="chain_name=='F'") # Best quality monomer

### 2. Structural Alignment
We move the high-quality barstar (F) to the bound position of barstar E using a least-RMSD fit. The fit selection must match between both systems and contain at least three non-collinear atoms; the C-alpha selection used here satisfies that geometric requirement.

In [ ]:
barstar_aligned = msm.structure.align(barstar_F, selection='atom_name=="CA"',
                                      reference_molecular_system=barstar_E)

msm.view([barnase, barstar_aligned])

### 3. Merging and Curing
We merge them into a single system and repair missing atoms or alternate locations.

In [ ]:
complex = msm.merge([barnase, barstar_aligned])

# Check and fix missing heavy atoms
complex = msm.build.add_missing_heavy_atoms(complex)
complex = msm.build.add_missing_hydrogens(complex, pH=7.4)

msm.info(complex)

### 4. Persistence
Finally, we save our masterpiece as a clean PDB file.

In [ ]:
msm.convert(complex, to_form='barnase_barstar_cured.pdb')

```{key-takeaway}
MolSysMT allows you to perform structural surgery: harvest, align, merge, and repair systems from disparate sources in a single script.
```